In [ ]:
import os, subprocess
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_req = f"/Workspace{os.path.dirname(os.path.dirname(_nb))}/requirements-ingestion.txt"
subprocess.check_call(["pip", "install", "-q", "-r", _req])
dbutils.library.restartPython()

# Ask Jorge — Document Ingestion

Extracts text from PDF files, cleans it, chunks it by token count, and writes
chunks to `workspace.default.jorge_cv_chunks` (Delta + CDF enabled).

**Parameters:**
- `file_path` — full path to the file in the Databricks Volume
- `full_reindex` — `'true'` truncates the whole table before writing; `'false'` replaces only chunks for this file

In [ ]:
import os
import re
import unicodedata
from datetime import datetime, timezone
from typing import List, Tuple

import pypdf
from transformers import AutoTokenizer

TABLE_NAME         = "jorge.cv_rag.jorge_cv_chunks"
TOKENIZER_CACHE    = "/tmp/hf_cache"
MAX_TOKENS         = 500
OVERLAP_TOKENS     = 50   # tokens shared between consecutive chunks for context continuity
MIN_CHUNK_TOKENS   = 30   # discard fragments shorter than this
SEPARATORS         = ["\n\n", "\n", ". ", " ", ""]

In [ ]:
dbutils.widgets.text("file_path",    "")
dbutils.widgets.text("full_reindex", "true")
file_path    = dbutils.widgets.get("file_path")
full_reindex = dbutils.widgets.get("full_reindex").lower() == "true"

print(f"file_path:    {file_path}")
print(f"full_reindex: {full_reindex}")

In [ ]:
def clean_text(text: str) -> str:
    """Normalise and clean raw PDF text."""
    # Unicode normalisation (e.g. ligatures: ﬁ → fi)
    text = unicodedata.normalize("NFKC", text)
    # Remove BPE special tokens that leak into some PDFs
    text = re.sub(r"</?s>", "", text)
    # Non-breaking spaces → regular space
    text = text.replace("\xa0", " ")
    # Rejoin words split by hyphen at end-of-line
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    # Collapse runs of spaces/tabs (keep newlines for separator logic)
    text = re.sub(r"[ \t]+", " ", text)
    # Collapse 3+ blank lines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Remove lone page numbers (lines with only digits)
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)
    # Strip leading/trailing whitespace per line
    text = "\n".join(line.strip() for line in text.splitlines())
    return text.strip()


def extract_pages(pdf_path: str) -> List[Tuple[int, str]]:
    """Return list of (page_number, cleaned_text) tuples."""
    pages = []
    try:
        with open(pdf_path, "rb") as f:
            reader = pypdf.PdfReader(f)
            for i, page in enumerate(reader.pages, start=1):
                raw = page.extract_text() or ""
                cleaned = clean_text(raw)
                if cleaned:
                    pages.append((i, cleaned))
    except Exception as e:
        raise RuntimeError(f"Failed to read {pdf_path}: {e}") from e
    return pages


def chunk_page(text: str, tokenizer, page_num: int) -> List[dict]:
    """
    Split a page's text into overlapping chunks under MAX_TOKENS.
    Returns list of dicts with chunk_text, page_number, chunk_index, chunk_tokens.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        end = min(start + MAX_TOKENS, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()
        if len(chunk_tokens) >= MIN_CHUNK_TOKENS and chunk_text:
            chunks.append({
                "chunk_text":   chunk_text,
                "page_number":  page_num,
                "chunk_index":  idx,
                "chunk_tokens": len(chunk_tokens),
            })
            idx += 1
        if end == len(tokens):
            break
        # Advance by (MAX_TOKENS - OVERLAP_TOKENS) to create overlap
        start += MAX_TOKENS - OVERLAP_TOKENS
    return chunks


def ensure_table():
    """Create or truncate the chunks table."""
    try:
        spark.sql(f"DESCRIBE TABLE {TABLE_NAME}")
        if full_reindex:
            print(f"full_reindex=True: truncating {TABLE_NAME}")
            spark.sql(f"TRUNCATE TABLE {TABLE_NAME}")
        spark.sql(
            f"ALTER TABLE {TABLE_NAME} "
            "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    except Exception:
        print(f"Creating {TABLE_NAME}...")
        spark.sql(f"""
            CREATE TABLE {TABLE_NAME} (
                id           BIGINT GENERATED BY DEFAULT AS IDENTITY,
                chunk_text   STRING  NOT NULL,
                source_file  STRING  NOT NULL,
                page_number  INT,
                chunk_index  INT,
                chunk_tokens INT,
                ingested_at  TIMESTAMP
            )
            TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"Table {TABLE_NAME} created.")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "hf-internal-testing/llama-tokenizer",
    cache_dir=TOKENIZER_CACHE,
)

ensure_table()

filename = os.path.basename(file_path)
print(f"Extracting pages from {filename}...")
pages = extract_pages(file_path)
if not pages:
    raise ValueError(f"No text extracted from {file_path}. Check file format.")
print(f"Extracted {len(pages)} pages.")

# If not full reindex, delete existing chunks for this file before re-inserting
if not full_reindex:
    print(f"Deleting existing chunks for {filename}...")
    spark.sql(f"DELETE FROM {TABLE_NAME} WHERE source_file = '{filename}'")

all_chunks = []
now = datetime.now(timezone.utc)
for page_num, page_text in pages:
    page_chunks = chunk_page(page_text, tokenizer, page_num)
    for c in page_chunks:
        all_chunks.append((
            c["chunk_text"],
            filename,
            c["page_number"],
            c["chunk_index"],
            c["chunk_tokens"],
            now,
        ))

print(f"Total chunks: {len(all_chunks)} across {len(pages)} pages.")

schema = "chunk_text STRING, source_file STRING, page_number INT, chunk_index INT, chunk_tokens INT, ingested_at TIMESTAMP"
(
    spark.createDataFrame(all_chunks, schema=schema)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLE_NAME)
)

# Summary
total = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_NAME}").collect()[0]["n"]
print(f"Done. {len(all_chunks)} chunks written. Table total: {total} chunks.")